# Fine-tuning

> LoRA adapters for Qwen3.5 0.8B and 2B, trained on Haiku's messages, and the learning curve

In [ ]:
#| default_exp finetune

Fine-tuning trains LoRA adapters with [mlx-lm](https://github.com/ml-explore/mlx-lm) on Haiku's messages. Each model is trained on 50, 100 and 199 examples, and scored on the frozen test set with the code checks, like the [baseline](05_baseline.ipynb); the scores against the number of examples form the learning curve of the [scope](design/scope.ipynb).

In [ ]:
#| export
import json, math, random, subprocess, sys
from pathlib import Path
from edgeslm.teacher import user_prompt
from edgeslm.dataset import read_jsonl, write_jsonl
from edgeslm.baseline import run_model

In [ ]:
#| hide
from fastcore.test import *

## Training data

`teacher_rows` keeps Haiku's messages that passed every check, in one fixed random order. A training set of `n` examples is the first `n` rows, so each smaller set is part of the larger ones and the curve reflects size, not a different draw:

In [ ]:
#| export
TEACHER = 'claude-haiku-4-5'

def teacher_rows(
    rows:list, # Labelled rows from `label_all`
    seed:int=0, # Seed of the fixed order
)->list:
    "The rows by `TEACHER` that pass every check, in a fixed random order"
    keep = sorted((r for r in rows if r['model'] == TEACHER and all(r['checks'].values())), key=lambda r: r['id'])
    random.Random(seed).shuffle(keep)
    return keep

In [ ]:
train = teacher_rows(read_jsonl('../data/train-v1.jsonl'))
valid = teacher_rows(read_jsonl('../data/valid-v1.jsonl'))
len(train), len(valid)

(199, 30)

The fine-tuned models get a one-line instruction, `SHORT_PROMPT`, instead of the baseline's student prompt, for two reasons. Training on Qwen3.5 with the student prompt ran out of memory on a 32 GB M5, even one example at a time: an example with the full prompt is 1,304 tokens, and the memory that training needs grows steeply with length. With the short prompt an example is about 165 tokens and trains easily. On the board, a short prompt also saves reading 1,300 tokens before every message. The model learns the rules from the examples instead of reading them, which is what fine-tuning is for.

mlx-lm reads chat rows: the instruction as the system message, the record as the user message, and the teacher's message as the reply. Qwen3.5's chat template writes the reply after the same empty thinking block that `student_input` ends with, so training text starts exactly with the text the model sees at inference:

In [ ]:
#| export
SHORT_PROMPT = 'Write the WhatsApp message for this triage record.'

def chat_row(row:dict)->dict:
    "A labelled row as an mlx-lm chat example"
    return dict(messages=[dict(role='system', content=SHORT_PROMPT), dict(role='user', content=user_prompt(row['record'])),
                          dict(role='assistant', content=row['message'])])

def write_mlx_data(
    path, # Directory for `train.jsonl` and `valid.jsonl`
    train:list, # Labelled training rows
    valid:list, # Labelled validation rows
):
    "Write `train` and `valid` in mlx-lm's chat format"
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    for name, rows in (('train', train), ('valid', valid)):
        (path/f'{name}.jsonl').write_text(''.join(json.dumps(chat_row(r), ensure_ascii=False) + '\n' for r in rows))

## Training

`lora_cmd` builds the command that runs mlx-lm's trainer as a separate process. Every run uses the same settings: mlx-lm's default rank, layers and maximum length, a learning rate of 1e-4, and 4 epochs. Training Qwen3.5 in mlx-lm takes far more memory than a model of this size should: batches of 4 short examples, about 1,200 tokens, ran out of memory on a 32 GB M5. Each step therefore takes one example, with gradient checkpointing, and the weights update every 4 steps, which matches batches of 4 for the optimiser. Only the reply counts in the loss, and validation runs once per epoch:

In [ ]:
#| export
def lora_cmd(
    model_id:str, # Hugging Face id of the MLX model
    data:str, # Directory written by `write_mlx_data`
    adapters:str, # Directory for the trained adapters
    n:int, # Number of training examples
    accum:int=4, # Examples per weight update, one at a time
    epochs:int=4, # Passes over the training set
    lr:float=1e-4, # Learning rate
)->list: # The command, as arguments for `subprocess.run`
    "The mlx-lm command that trains LoRA adapters for `model_id` on `data`"
    return [sys.executable, '-m', 'mlx_lm', 'lora', '--model', model_id, '--train', '--data', str(data), '--adapter-path', str(adapters),
            '--mask-prompt', '--batch-size', '1', '--grad-accumulation-steps', str(accum), '--grad-checkpoint',
            '--iters', str(epochs * n), '--learning-rate', str(lr), '--steps-per-eval', str(n), '--val-batches', '-1', '--seed', '0']

This cell trained the six adapters and scored each one on the test set, and does not run in tests. On an Apple M5 with 32 GB, training took about 2 hours and scoring about 80 minutes. The 2B on 199 examples took 65 minutes to train and 37 to score, against 13 and 12 for the 2B on 100 examples. We have not found the cause of that slowdown yet:

In [ ]:
#| eval: false
test = read_jsonl('../data/test-v1.jsonl')
MODELS = {'qwen3.5-0.8b': 'mlx-community/Qwen3.5-0.8B-bf16', 'qwen3.5-2b': 'mlx-community/Qwen3.5-2B-bf16'}
for name, mid in MODELS.items():
    for n in (50, 100, 199):
        run = f'{name}-n{n}'
        write_mlx_data(f'../models/{run}/data', train[:n], valid)
        res = subprocess.run(lora_cmd(mid, f'../models/{run}/data', f'../models/{run}/adapters', n), capture_output=True, text=True, check=True)
        Path(f'../models/{run}/train.log').write_text(res.stdout)
        write_jsonl(f'../data/finetune-v1-{run}.jsonl', run_model(mid, test, adapter_path=f'../models/{run}/adapters', sp=SHORT_PROMPT))

## Results

In [ ]:
from edgeslm.baseline import score_table, markers

The table puts each untuned baseline (`base`, with the two-example student prompt) beside its three fine-tuned runs (`n50`, `n100`, `n199`, with `SHORT_PROMPT`). It compares the best prompt we found for the untuned models with the setup we intend to deploy. It is not a controlled test of the prompt:

In [ ]:
test = read_jsonl('../data/test-v1.jsonl')
runs = {f'{m} {k}': read_jsonl(f'../data/baseline-v2-qwen3.5-{m}.jsonl' if k == 'base' else f'../data/finetune-v1-qwen3.5-{m}-{k}.jsonl')
        for m in ('0.8b', '2b') for k in ('base', 'n50', 'n100', 'n199')}
score_table(runs)

| Check | 0.8b base | 0.8b n50 | 0.8b n100 | 0.8b n199 | 2b base | 2b n50 | 2b n100 | 2b n199 |
|---|---|---|---|---|---|---|---|---|
| all | 123 / 270 | 78 / 270 | 255 / 270 | 264 / 270 | 70 / 270 | 229 / 270 | 260 / 270 | 250 / 270 |
| format | 269 / 270 | 217 / 270 | 270 / 270 | 270 / 270 | 270 / 270 | 270 / 270 | 270 / 270 | 270 / 270 |
| coverage | 269 / 270 | 151 / 270 | 259 / 270 | 270 / 270 | 269 / 270 | 259 / 270 | 267 / 270 | 260 / 270 |
| no_invented | 270 / 270 | 186 / 270 | 261 / 270 | 270 / 270 | 270 / 270 | 270 / 270 | 269 / 270 | 267 / 270 |
| severity | 143 / 270 | 253 / 270 | 266 / 270 | 270 / 270 | 79 / 270 | 269 / 270 | 269 / 270 | 262 / 270 |
| no_numbers | 269 / 270 | 113 / 270 | 262 / 270 | 270 / 270 | 264 / 270 | 270 / 270 | 269 / 270 | 267 / 270 |
| length | 267 / 270 | 237 / 270 | 268 / 270 | 264 / 270 | 266 / 270 | 267 / 270 | 264 / 270 | 264 / 270 |
| amount | 234 / 270 | 215 / 270 | 270 / 270 | 270 / 270 | 237 / 270 | 242 / 270 | 270 / 270 | 270 / 270 |

With 100 examples, both models pass every check in 255 or more of the 270 messages, against 123 for the untuned 0.8B and 70 for the untuned 2B. With 199 examples the 0.8B reaches 264, and the 2B falls back to 250.

With 50 examples, the 0.8B scores below its baseline. It mixes up locations and repeats them ("plant 2, plant 2 ... plants 2, 2 and 3"), 41 of its messages repeat one phrase until the token limit, and some contain a stray `</think>` tag. The 2B on 50 examples has none of these failures and passes 229.

We scored the adapter saved at the end of the fourth epoch. Validation loss after each epoch, from `models/<run>/train.log`:

| Run | Epoch 1 | Epoch 2 | Epoch 3 | Epoch 4 |
|---|---|---|---|---|
| 0.8B, 50 | 0.327 | 0.232 | 0.166 | 0.218 |
| 0.8B, 100 | 0.314 | 0.126 | 0.118 | 0.097 |
| 0.8B, 199 | 0.132 | 0.090 | 0.071 | 0.081 |
| 2B, 50 | 0.281 | 0.254 | 0.177 | 0.133 |
| 2B, 100 | 0.276 | 0.166 | 0.122 | 0.107 |
| 2B, 199 | 0.101 | 0.096 | 0.062 | 0.104 |

The 2B on 199 examples has the largest rise in the last epoch, from 0.062 to 0.104. Its drop in score is consistent with overfitting in that epoch. Scoring its third-epoch adapter would test this.

Fine-tuning teaches the severity-to-marker mapping that the prompt did not. The 0.8B on 199 examples writes the right marker for all 469 test items. In the pairs below, `None` marks an item whose location the model wrote differently from the record:

In [ ]:
markers(runs['0.8b n199'], test), markers(runs['2b n100'], test)

(Counter({('early', '👀'): 209, ('moderate', '⚠️'): 148, ('severe', '🚨'): 112}),
 Counter({('early', '👀'): 207,
          ('moderate', '⚠️'): 147,
          ('severe', '🚨'): 111,
          (None, '⚠️'): 1,
          ('severe', '⚠️'): 1}))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()